# Konya | Data and spatial diagnostics
CME434 · Karabük University

Inspect the released subset and reproduce feature joins, Moran’s I and LISA without fetching new data.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
ART = ROOT / 'artifacts'


In [ ]:
from spatial import SpatialLayers, FEATURES, block_key
from modeling import moran_statistics

raw = pd.read_csv(ROOT / 'data' / 'value_points.csv')
training = pd.read_csv(ROOT / 'data' / 'training_points.csv')
assert len(raw) == 302 and len(training) == 300
training['unit_tl_m2'] = training.price_tl / training.area_m2
display(training.groupby('imar_durumu').unit_tl_m2.agg(['size', 'median']).sort_values('size', ascending=False))


In [ ]:
layers = SpatialLayers()
joined = layers.enrich(training)
joined['cv_block'] = block_key(joined)
assert layers.contains(joined.longitude, joined.latitude).all()
display(joined[list(FEATURES)].isna().sum().rename('missing'))


In [ ]:
results, lisa = moran_statistics(joined, {'unit_tl_m2': joined.unit_tl_m2.to_numpy()})
display(results)
display(lisa.groupby('lisa_fdr_05').size())
display(Image(filename=str(ART / 'figures' / '02_konya_lisa.png'),width=850))


The private academic archive contains the original source references and the complete regression join. The public notebook uses the screened and reduced listing table.